# Code Notebooks and the TPR-DB

## Module 01: Lesson 01

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Critt-Kent/CRITT-Academy/blob/main/modules/01_Foundations/02_Notebooks_and_TPRDB.ipynb)

### What You Will Do In This Lesson

1. Make sure your Jupyter kernel is set up
2. Install and import all necessary Python libraries
3. Learn how to use the `tprdb-utilities` function called `fetch_TPRDB_tables()`
4. Learn how to use the `tprdb-utilities` function called `read_TPRDB_tables()`

## Step 1. Make sure your Jupyter kernel is running

Before executing any code in this notebook, please verify that your Python environment is active:

1. Look at the **top-right corner** of this window. 
2. Ensure it displays an active environment name or a version of Python (e.g., `Python 3.x`).
3. If it reads **Select Kernel** or **No Kernel**, click that text block and select the default **Python 3 (ipykernel)** or your preferred local environment from the dropdown menu.

*Once your kernel is connected (indicated by a solid or hollow circle next to the Python name in the top-right), you are ready for step 2.*

### Jupyter Basics

This first lesson is not meant to teach you how to use Jupyter notebooks, but here are some YouTube videos that might be helpful if you've never used a code notebook before:

- [Installation Guide and Basics](https://youtu.be/7oNFaX2q5uU?si=liiHpa_noI3_QQvo)
- [Common Mistakes and Essential Shortcuts](https://youtu.be/VJbL5LSqGDc?si=qG0bCjQaSlmsizRq)
- [Code Cells and Markdown Cells](https://youtu.be/Tlv94Z0lGpo?si=7v0eb5RHtF0IXnH0)

## Step 2. Install and import all necessary Python libraries

**Python libraries** are just packages of code that are used for a specific purpose. Here are the Python libraries that are used by a typical CRITT Academy lesson:

- **NumPy**: `numpy` is used for scientific computing and data science
- **Pandas**: `pandas` is used for data manipulation, organization, and analysis
- **SciPy `stats`**: the `stats` module within `scipy` is used for statistical analysis and probability distributions
- **Matplotlib**: `matplotlib` is used for data visualization
- **Seaborn**: `seaborn` is used for nice-looking statistical graphics
- **tprdb-utilities**: `tprdb_utilities` is used for getting data remotely from the CRITT TPR-DB and then for reading data tables from a certain study (or studies) into a Pandas DataFrame.

### So what does the following code in step 2 do?

The code in this step is something you will see in all CRITT Academy lessons.

1. First, you make sure you have all the Python libraries installed that will be necessary for the lesson.
2. Then, you *import* the Python libraries so they are ready to use for the entire rest of the notebook


In [ ]:
# Environment Setup
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

# Enforce a minimum Python version of 3.9+
if sys.version_info < (3, 9):
    raise RuntimeError(
        f"❌ Python 3.9 or higher is required. "
        f"You are currently running Python {sys.version_info.major}.{sys.version_info.minor}."
    )

REQUIRED = {
    "numpy": "2.4.0",
    "pandas": "3.0.0",
    "scipy": "1.17.0",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.0",
    "tprdb-utilities": "2.0.0",
}

def _check_versions():
    outdated = []
    for pkg, min_ver in REQUIRED.items():
        try:
            installed = version(pkg)
            if Version(installed) < Version(min_ver):
                outdated.append(f"  {pkg}: installed {installed}, need >={min_ver}")
        except PackageNotFoundError:
            raise ImportError(f"{pkg} is not installed")
    return outdated

# Check if dependencies are missing and install them automatically
try:
    import numpy as np
    import pandas as pd
    import scipy.stats
    import matplotlib as mpl
    import seaborn as sns
    import tprdb_utilities as tpr

    outdated = _check_versions()
    if outdated:
        raise ImportError("Outdated packages:\n" + "\n".join(outdated))
    else:
        print("🤠 All core dependencies are already installed. You are ready to go! 🤘")
except ImportError:
    print("Missing dependencies detected. Installing required packages...")
    # The %pip magic ensures installation happens in the active Jupyter kernel
    %pip install -r https://raw.githubusercontent.com/Critt-Kent/CRITT-Academy/main/requirements.txt
    
    print("\n🤠 Installation complete 🤘 If imports fail on the next cell, please restart the kernel.")

Missing dependencies detected. Installing required packages...
Note: you may need to restart the kernel to use updated packages.

🤠 Installation complete 🤘 Please restart the kernel if imports fail on the next cell.


In [4]:
# Now, import the libraries

# Standard Python library imports
import numpy as np
import pandas as pd
import scipy.stats
import matplotlib as mpl
import seaborn as sns

# TPR-DB utilities import
from tprdb_utilities import fetch_TPRDB_tables, read_TPRDB_tables

## Step 3. Fetch some data tables from the CRITT TPR-DB

Data in the CRITT TPR-DB is living on a server in Kent, Ohio. If we want to analyze that data, we first need to ask ourselves where we are.

### On the CRITT Jupyter server?

If you are using Jupyter from the CRITT Jupyter server, then you are already in the same place as the data and you can skip all of step 3 (this step) and move on to step 4 (the next step).

### Are you somewhere else?

If you are **not** on the CRITT Jupyter server, then this step is for you. Since the data is on a server that you are not working from, you need to download it.

You will use the `fetch_TPRDB_tables()` function from the `tprdb-utilities` library to download some data tables to wherever you currently are (your own computer, another Jupyter server, Google Colab, etc.). If you'd like to read the full documentation for this library and its functions, you can find it on the [Python Package Index](https://pypi.org/project/tprdb-utilities/), but we'll explain it simply right here:

The `fetch_TPRDB_tables()` function is very simple:
1. First, you tell it
    1. `path`: **where to save** the downloaded data
    2. `StudyID`: what TPR-DB **study** you would like to download from
    3. `extensions`: the data table **extension(s)** (one or several) you want to download
    4. **whether or not** the study you're downloading from is **public** (this lesson will only cover downloading from a public study)
2. Then, the function
    1. creates a folder called `tprdb-mothership-clone` in the location you gave it
    2. saves the data tables in the mothership clone folder

#### Help Docs

Before actually using the `fetch_TPRDB_tables()` function, you can see the help documentation by using the `?` shorthand character after the function name.

In [8]:
fetch_TPRDB_tables?

Signature:
fetch_TPRDB_tables(
    path,
    StudyID,
    extension,
    public,
    username=None,
    token=None,
    verbose=0,
)
Docstring:
Download TPR-DB data tables from the CRITT TPR-DB API and save them
locally in a directory structure that mirrors the TPR-DB server layout.

Makes one HTTP GET request per extension to the CRITT TPR-DB REST API,
receives a ``.zip`` archive, and extracts its contents directly into the
appropriate ``Tables/`` subdirectory.  The resulting file structure is
identical to the layout expected by ``read_TPRDB_tables``, so the two
functions are designed to be used in sequence.

**File structure created**::

    <path>/
    └── tprdb-mothership-clone/
        ├── PUBLIC/                 ← public studies
        │   └── <StudyID>/
        │       ├── studySummary.xml
        │       └── Tables/
        │           ├── session1.<ext>
        │           └── ...
        └── <username>/             ← private studies (when public=False)
            └── <Study

#### The **mothership clone**

It is important to use the same `path` variable for your "mothership clone" every time you use the `read_TPRDB_tables()` function because it will recognize if the data you want has already been downloaded and will conserve resources by not downloading it twice. It will also recognize if the CRITT TPR-DB has a more up-to-date version of the data tables than you already have in your mothership clone and will replace them with the more up-to-date version.

**Change the value of the `mothership_clone_location variable` below, and then we will be ready to run the `read_TPRDB_tables()` function**

In [6]:
# Change what is in the quotes to point to where you want your mothership clone to be saved.
mothership_clone_location = "./"

In [7]:
# fetch data tables (notice we are using the `mothership_clone_location` variable we defined in the cell above)

fetch_TPRDB_tables(
    path=mothership_clone_location,
    StudyID="SG12",
    extension=["ss", "sg", "st"],
    public=True,
)

=== fetch_TPRDB_tables Summary ===
StudyID  : SG12
Clone dir: ./tprdb-mothership-clone
User dir : PUBLIC

Extension  Status            Time
---------  ----------------  ------
ss         Up to date (304)  0.18s
sg         Up to date (304)  0.19s
st         Up to date (304)  0.16s

To read these files with read_TPRDB_tables:
  path      = "./tprdb-mothership-clone"
  user      = "PUBLIC"
  studies   = ["SG12"]


## Step 4: Read the tables into a DataFrame

Now you will use the `read_TPRDB_tables()` function to read a certain data table from one study into a DataFrame so that it can be analyzed.

### Help Docs

Once again, before you run the function, look at its help documentation:

In [ ]:
read_TPRDB_tables?

Once again, where we are running this notebook from matters:

### On the CRITT Jupyter server?

If this is the case, then you should copy and paste the following code into the next cell:

```py
sessions = read_TPRDB_tables(
    studies=["SG12"],
    extension="ss",
    mothership=True,
    user="PUBLIC",
    verbose=1,
)
```

### Are you somewhere else?

In this case, you need to use your mothership clone you created before, so you should copy and paste the following code into the next cell (notice that we define the `path` of the mothership clone folder using the `mothership_clone_location` variable we declared in Step 3):

```py
sessions = read_TPRDB_tables(
    studies=["SG12"],
    extension="ss",
    mothership=False,
    path=f"{mothership_clone_location}/tprdb-mothership-clone",
    user="PUBLIC",
    verbose=1,
)
```

In [ ]:
# Paste correct code from above depending on your situation, and then run the cell.